# Generation script

In [9]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.qasm2 import dumps
import numpy as np
import os
import time
import datetime
import json
import platform
import psutil

def get_system_info():
    info = {
        "platform": f"{platform.system()} {platform.release()}",
        "architecture": platform.machine(),
        "processor": platform.processor(),
        "cpu_count": os.cpu_count(),
        "total_ram_gb": round(psutil.virtual_memory().total / (1024**3), 2),
        "available_ram_gb": round(psutil.virtual_memory().available / (1024**3), 2)
    }
    try:
        import cpuinfo
        cpu = cpuinfo.get_cpu_info()
        info["cpu_brand"] = cpu.get('brand_raw', 'N/A')
        info["cpu_hz"] = cpu.get('hz_advertised_friendly', 'N/A')
    except ImportError:
        info["cpu_brand"] = 'N/A'
        info["cpu_hz"] = 'N/A'
    return info

def generate_random_clifford_t_circuit(num_qubits, h_count, t_count, s_count=10, z_count=25, cz_count=10):
    qc = QuantumCircuit(num_qubits)
    # Prepare GHZ state on all qubits
    qc.h(0)
    for i in range(num_qubits - 1):
        qc.cx(i, i + 1)
    # Add random Clifford+T gates
    for _ in range(h_count):
        qc.h(np.random.randint(0, num_qubits))
    for _ in range(s_count):
        qc.s(np.random.randint(0, num_qubits))
    for _ in range(z_count):
        qc.z(np.random.randint(0, num_qubits))
    t_qubits = np.random.choice(num_qubits, size=t_count, replace=True)
    for qubit in t_qubits:
        qc.t(qubit)
    for _ in range(cz_count):
        control = np.random.randint(0, num_qubits)
        target = np.random.randint(0, num_qubits)
        while target == control:
            target = np.random.randint(0, num_qubits)
        qc.cz(control, target)
    return qc

np.random.seed(42)

# Parameters
qubit_sizes = list(range(10, 30, 5))  # 10, 15, 20, 25, 30
t_gate_counts = list(range(0, 50, 10))  # 0, 10, 20, 30, 40, 50
h_gate_counts = list(range(5, 20, 5))  # 5, 10, 15, 20
s_count = 10
z_count = 50
cz_count = 10

dataset_dirs = ["dataset", "../dataset"]
for base in dataset_dirs:
    os.makedirs(base, exist_ok=True)

results_log_path = "qiskit_aer_results.txt"
all_generated_files = []
simulation_times = []
system_infos = []

# Main loop
for num_qubits in qubit_sizes:
    for t_count in t_gate_counts:
        for h_count in h_gate_counts:
            qc = generate_random_clifford_t_circuit(num_qubits, h_count, t_count, s_count, z_count, cz_count)
            qc.save_statevector()
            filename_base = f"random_circuit_q{num_qubits}_h{h_count}_t{t_count}_s{s_count}_z{z_count}_cz{cz_count}"
            for base in dataset_dirs:
                d = f"{base}/{filename_base}"
                os.makedirs(d, exist_ok=True)
                # Export to QASM2
                qasm_output = dumps(qc)
                qasm_filename = f"{d}/{filename_base}.qasm"
                with open(qasm_filename, 'w') as f:
                    f.write(qasm_output)
                all_generated_files.append(qasm_filename)
                # Simulate and record time
                simulator = AerSimulator(method='statevector')
                start_sim = time.time()
                simulation_failed = False
                try:
                    result = simulator.run(qc).result()
                    elapsed_sim = time.time() - start_sim
                    # Save statevector as raw binary if available
                    statevector = result.get_statevector(0)
                    statevector_bin_filename = f"{d}/{filename_base}_statevector.bin"
                    statevector.data.astype(np.complex128).tofile(statevector_bin_filename)
                    all_generated_files.append(statevector_bin_filename)
                except Exception as e:
                    elapsed_sim = 0
                    simulation_failed = True
                # Save siminfo as JSON
                siminfo = {
                    "simulation_time_seconds": elapsed_sim,
                    "num_qubits": num_qubits,
                    "h_count": h_count,
                    "t_count": t_count,
                    "filename_base": filename_base,
                    "simulation_failed": simulation_failed
                }
                siminfo_filename = f"{d}/{filename_base}_siminfo.json"
                with open(siminfo_filename, 'w') as f:
                    json.dump(siminfo, f, indent=2)
                all_generated_files.append(siminfo_filename)
                simulation_times.append({
                    "filename_base": filename_base,
                    "simulation_time_seconds": elapsed_sim,
                    "simulation_failed": simulation_failed
                })

# Write results to log file in requested order
with open(results_log_path, "w") as log_file:
    log_file.write("Generated files:\n" + "\n".join(all_generated_files) + "\n\n")
    log_file.write("Simulation times (seconds):\n")
    for entry in simulation_times:
        log_file.write(f"{entry['filename_base']}: {entry['simulation_time_seconds']:.4f} (failed: {entry['simulation_failed']})\n")
    log_file.write("\nSystem info:\n")
    log_file.write(json.dumps(get_system_info(), indent=2) + "\n")

In [8]:
import shutil
import os

def clean_dataset_folders():
    # Remove all contents of 'dataset' and '../dataset' folders
    for folder in ['dataset', '../dataset']:
        if os.path.exists(folder):
            for entry in os.listdir(folder):
                entry_path = os.path.join(folder, entry)
                if os.path.isdir(entry_path):
                    shutil.rmtree(entry_path)
                else:
                    os.remove(entry_path)
    print("All contents of 'dataset' and '../dataset' have been deleted.")

# Usage:
clean_dataset_folders()

All contents of 'dataset' and '../dataset' have been deleted.


# clean up script